In [ ]:
!nvcc --version


In [ ]:
!nvidia-smi


In [ ]:
import torch

print(torch.version.cuda)


In [ ]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))


In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128


In [ ]:
import torch

print(torch.cuda.is_available())


In [ ]:
%pip install -U ultralytics


In [ ]:
%pip install -e '.[dev]'


In [ ]:
import torch

torch.cuda.is_available()


In [ ]:
import dlib
import face_recognition
import torch
import ultralytics

print(f"Dlib: {dlib.__version__}")
print(f"Face-Recognition: {face_recognition.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

import mediapipe

print(f"mediapipe: {mediapipe.__version__}")

import pygame

print(f"pygame: {pygame.__version__}")


In [ ]:
!yolo solutions inference

!yolo solutions inference model="yolo12s.pt"


In [ ]:
from ultralytics import solutions

inf = solutions.Inference(
    model="yolo11n.pt",  # you can use any model that Ultralytics supports, e.g., YOLO11, YOLOv10
)

inf.inference()

# Make sure to run the file using command `streamlit run path/to/file.py`


In [ ]:
import cv2
from numpy import source

from ultralytics import solutions
from ultralytics.utils.plotting import Annotator

import os
import cv2
import numpy as np
import face_recognition
import pygame

# from ultralytics import solutions
from ultralytics import YOLO
from ultralytics.solutions.config import SolutionConfig
from ultralytics.utils import LOGGER

from ultralytics.solutions.solutions import BaseSolution, SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors

# ========== 🔊 SOUND SETUP ==========
pygame.mixer.init()
ALARM_FILE = "../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"
if os.path.exists(ALARM_FILE):
    pygame.mixer.music.load(ALARM_FILE)
else:
    print(f"[WARNING] Alarm file '{ALARM_FILE}' not found.")


# ========== 🧠 KNOWN FACE ENCODING LOADER ==========
KNOWN_FACE_DIR = "../family_members/"
known_face_encodings, known_face_names = [], []

if os.path.exists(KNOWN_FACE_DIR):
    for name in os.listdir(KNOWN_FACE_DIR):
        person_dir = os.path.join(KNOWN_FACE_DIR, name)
        if not os.path.isdir(person_dir):
            continue
        for filename in os.listdir(person_dir):
            path = os.path.join(person_dir, filename)
            try:
                img = face_recognition.load_image_file(path)
                enc = face_recognition.face_encodings(img)
                if enc:
                    known_face_encodings.append(enc[0])
                    known_face_names.append(name)
                    print(f"[INFO] Loaded face for {name} from {filename}")
            except Exception as e:
                print(f"[ERROR] Failed loading {path}: {e}")
else:
    print("[WARNING] No known_faces directory found.")


# ========== 👁️ FACE-RECOGNITION ALARM (REVISED & OPTIMIZED) ==========
class FaceRecognitionAlarmVisionEye(solutions.VisionEye):
    def __init__(self, *args, known_face_encodings=None, known_face_names=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.known_face_encodings = known_face_encodings or []
        self.known_face_names = known_face_names or []
        self.sound_played = False
        # Best practice: Set face recognition tolerance during initialization
        self.face_tolerance = 0.55
        self.vision_point = self.CFG["vision_point"]
        self.records = self.CFG.get("records", 1)
        # self.show = self.CFG.get("show", True)

    def play_sound(self):
        """Plays the alarm sound if it's not already playing."""
        if not self.sound_played:
            if pygame.mixer.get_init() and not pygame.mixer.music.get_busy():
                pygame.mixer.music.play()
                self.sound_played = True
                LOGGER.info("🚨 Alarm Triggered: Unknown person count reached threshold.")

    def reset_sound(self):
        """Stops the alarm sound and resets the state."""
        if self.sound_played:
            if pygame.mixer.get_init():
                pygame.mixer.music.stop()
            self.sound_played = False
            LOGGER.info("🟢 Alarm Reset: Area clear.")

    def __call__(self, im0):
        """
        Processes a single frame for person detection and face recognition.
        This implementation follows best practices for accuracy and performance.
        """
        # 1. Get person detections from the base class
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, line_width=self.line_width)

        unknown_person_count = 0

        # 2. Optimize by finding all faces in the frame at once (on a smaller version)
        # This is much faster than processing crops for each person.
        h, w, _ = im0.shape
        small_frame = cv2.resize(im0, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        # 3. Iterate through detected PERSONS from YOLO
        for box, conf, cls, t_id in zip(self.boxes, self.confs, self.clss, self.track_ids):
            if int(cls) == 0:  # Person detection
                name = "Unknown"
                is_known = False

                # 4. Associate faces with person boxes
                # Check if any detected face is inside this person's bounding box
                person_box_left, person_box_top, person_box_right, person_box_bottom = map(int, box)

                for (face_top, face_right, face_bottom, face_left), face_encoding in zip(
                    face_locations, face_encodings
                ):
                    # Scale face locations back to original image size
                    face_top *= 4
                    face_right *= 4
                    face_bottom *= 4
                    face_left *= 4

                    # Check if the center of the face is inside the person's box
                    face_center_x = (face_left + face_right) // 2
                    face_center_y = (face_top + face_bottom) // 2

                    if (
                        person_box_left <= face_center_x <= person_box_right
                        and person_box_top <= face_center_y <= person_box_bottom
                    ):
                        # 5. Use robust face matching for the associated face
                        if self.known_face_encodings:
                            face_distances = face_recognition.face_distance(self.known_face_encodings, face_encoding)
                            best_match_index = np.argmin(face_distances)

                            if face_distances[best_match_index] < self.face_tolerance:
                                name = self.known_face_names[best_match_index]
                                is_known = True

                        # Once a face is matched to this person, stop checking other faces
                        break

                # 6. Determine label and color based on face recognition
                if not is_known:
                    unknown_person_count += 1
                    label = "Unknown"
                    box_color = colors(int(t_id), True)  # Use track-based color for unknown
                else:
                    label = f"{name}"
                    box_color = (0, 255, 0)  # Green for known persons

                # Build base label from the existing adjust_box_label()
                base_label = self.adjust_box_label(int(cls), float(conf) if conf is not None else 0.0, t_id)

                # Custom label for 'person' class (COCO id 0). Use CFG override if provided.
                prefix = str(self.CFG.get("person_label_prefix", label))
                custom_label = f"{prefix}:"
                final_label = f"{custom_label} {base_label}" if base_label else custom_label

                # Draw final label and vision eye mapping with determined color
                annotator.box_label(box, label=final_label, color=box_color)
            else:
                # For non-person classes, use default labeling with track-based color
                annotator.box_label(box, label=self.adjust_box_label(cls, conf, t_id), color=colors(int(t_id), True))

            annotator.visioneye(box, self.vision_point)

        # 7. Trigger alarm based on the COUNT of unknown people and the 'records' threshold
        if unknown_person_count >= self.records:
            self.play_sound()
        else:
            self.reset_sound()

        plot_im = annotator.result()
        self.display_output(plot_im)

        # Display track count on the frame
        total_tracks = len(getattr(self, "track_ids", []))
        cv2.putText(plot_im, f"Tracks: {total_tracks}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        return SolutionResults(plot_im=plot_im, total_tracks=len(self.track_ids))


if __name__ == "__main__":
    # cap = cv2.VideoCapture(0)
    cap = cv2.VideoCapture("../media_files/w")
    # cap = cv2.VideoCapture("../media_files/istockphoto-2002566174-640_adpp_is.mp4")
    # cap = cv2.VideoCapture("../media_files/WIN_20251103_14_11_20_Pro.mp4")
    # cap = cv2.VideoCapture("media_files/person/ruhama/VID_20251122_142652.mp4")
    # cap = cv2.VideoCapture("../media_files/w")
    # assert cap.isOpened(), "Error reading video file"

    # Video writer
    w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
    video_writer = cv2.VideoWriter("isegment_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    # Initialize vision eye object
    visioneyeInterface = FaceRecognitionAlarmVisionEye(
        show=True,  # display the output
        model="yolo11m.pt",  # use any model that Ultralytics support, i.e, YOLOv10
        # classes=[0, 19],  # generate visioneye view for specific classes
        vision_point=(20, 20),  # the point, where vision will view objects and draw tracks
        known_face_encodings=known_face_encodings,
        known_face_names=known_face_names,
        records=3,
        conf=0.3,
        iou=0.8,
        # show_labels=True,
    )


# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = visioneyeInterface(im0)

    print(results)  # access the output

    video_writer.write(results.plot_im)  # write the video file
    cv2.imshow("Face Recognition Alarm Vision Eye", results.plot_im)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
from ultralytics import solutions

# Define your monitoring zone
zone_pts = [(100, 100), (1100, 100), (1100, 600), (100, 600)]

# Initialize VisionEye with region support
# Note: You can customize the vision_point (e.g., center or bottom)
eye = solutions.VisionEye(model="yolo11m.pt", region=zone_pts, vision_point="center", show=True)

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Process filters tracks by the region automatically
    results = eye.process(frame)

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("isegment_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize instance segmentation object
isegment = solutions.InstanceSegmentation(
    show=True,  # display the output
    model="yolo11m-seg.pt",  # model="yolo11n-seg.pt" for object segmentation using YOLO11.
    # classes=[0, 2],  # segment specific classes, e.g., person and car with the pretrained model.
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = isegment(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
import torch
import torch.nn as nn

# Define a simple Autoencoder architecture
model = nn.Sequential(
    nn.Linear(64, 12),  # Encoder: Compress 64 features to 12
    nn.ReLU(),  # Non-linear activation
    nn.Linear(12, 64),  # Decoder: Reconstruct original 64 features
    nn.Sigmoid(),  # Output normalized between 0 and 1
)

# Create a dummy tensor simulating a flattened 8x8 image
input_data = torch.randn(1, 64)

# Perform the forward pass (encode and decode)
reconstruction = model(input_data)

print(f"Input shape: {input_data.shape}")  # torch.Size([1, 64])
print(f"Reconstructed shape: {reconstruction.shape}")  # torch.Size([1, 64])


In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
assert cap.isOpened(), "Error reading video file"

# Define region points
region_points = [(150, 150), (1130, 150), (1130, 570), (150, 570)]

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("trackzone_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Init trackzone (object tracking in zones, not complete frame)
trackzone = solutions.TrackZone(
    show=True,  # display the output
    region=region_points,  # pass region points
    model="yolo11n.pt",  # use any model that Ultralytics supports, e.g., YOLOv9, YOLOv10
    # line_width=2,  # adjust the line width for bounding boxes and text display
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = trackzone(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the video file

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("vision-eye-mapping.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Init vision eye object
visioneye = solutions.VisionEye(
    show=True,  # display the output
    model="yolo11n.pt",  # use any model that Ultralytics supports, e.g., YOLOv10
    # classes=[0, 2],  # generate visioneye view for specific classes
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = visioneye(im0)

    print(results)  # access the output

    video_writer.write(results.plot_im)  # write the video file

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
# Monitor objects position with visioneye
!yolo solutions visioneye show=True

# Pass a source video
!yolo solutions visioneye source="../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4"

# Monitor the specific classes
!yolo solutions visioneye classes="[0, 5]"


In [ ]:
import cv2

from ultralytics import solutions

# import cv2
# from numpy import source

from ultralytics import solutions
# from ultralytics.utils.plotting import Annotator

import os

# import cv2
import numpy as np
import face_recognition
import pygame

# from ultralytics import solutions
# from ultralytics import YOLO
# from ultralytics.solutions.config import SolutionConfig
from ultralytics.utils import LOGGER

from ultralytics.solutions.solutions import BaseSolution, SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors
import asyncio
import threading

# ========== 🔊 SOUND SETUP ==========
pygame.mixer.init()
ALARM_FILE = "../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"
if os.path.exists(ALARM_FILE):
    pygame.mixer.music.load(ALARM_FILE)
else:
    print(f"[WARNING] Alarm file '{ALARM_FILE}' not found.")


# ========== 🧠 KNOWN FACE ENCODING LOADER ==========
KNOWN_FACE_DIR = "../family_members/"
known_face_encodings, known_face_names = [], []

if os.path.exists(KNOWN_FACE_DIR):
    for name in os.listdir(KNOWN_FACE_DIR):
        person_dir = os.path.join(KNOWN_FACE_DIR, name)
        if not os.path.isdir(person_dir):
            continue
        for filename in os.listdir(person_dir):
            path = os.path.join(person_dir, filename)
            try:
                img = face_recognition.load_image_file(path)
                enc = face_recognition.face_encodings(img)
                if enc:
                    known_face_encodings.append(enc[0])
                    known_face_names.append(name)
                    print(f"[INFO] Loaded face for {name} from {filename}")
            except Exception as e:
                print(f"[ERROR] Failed loading {path}: {e}")
else:
    print("[WARNING] No known_faces directory found.")


class AiSecurityGuard(solutions.VisionEye):
    def __init__(self, *args, known_face_encodings=None, known_face_names=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.known_face_encodings = known_face_encodings or []
        self.known_face_names = known_face_names or []
        self.sound_played = False
        # Best practice: Set face recognition tolerance during initialization
        self.face_tolerance = 0.55
        self.vision_point = self.CFG["vision_point"]
        self.records = self.CFG.get("records", 1)
        # self.show = self.CFG.get("show", True)

    def play_sound(self):
        """Plays the alarm sound if it's not already playing."""
        if not self.sound_played:
            if pygame.mixer.get_init() and not pygame.mixer.music.get_busy():
                pygame.mixer.music.play()
                self.sound_played = True
                LOGGER.info("🚨 Alarm Triggered: Unknown person count reached threshold.")

    def reset_sound(self):
        """Stops the alarm sound and resets the state."""
        if self.sound_played:
            if pygame.mixer.get_init():
                pygame.mixer.music.stop()
            self.sound_played = False
            LOGGER.info("🟢 Alarm Reset: Area clear.")

    def __call__(self, im0):
        """
        Processes a single frame for person detection and face recognition.
        This implementation follows best practices for accuracy and performance.
        """
        # 1. Get person detections from the base class
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, self.line_width)

        unknown_person_count = 0

        # 2. Optimize by finding all faces in the frame at once (on a smaller version)
        # This is much faster than processing crops for each person.
        h, w, _ = im0.shape
        small_frame = cv2.resize(im0, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        # 3. Iterate through detected PERSONS from YOLO
        for cls, t_id, box, conf in zip(self.clss, self.track_ids, self.boxes, self.confs):
            if int(cls) == 0:  # Skip if not a person
                name = "Unknown"
                is_known = False

                # 4. Associate faces with person boxes
                # Check if any detected face is inside this person's bounding box
                person_box_left, person_box_top, person_box_right, person_box_bottom = map(int, box)

                for (face_top, face_right, face_bottom, face_left), face_encoding in zip(
                    face_locations, face_encodings
                ):
                    # Scale face locations back to original image size
                    face_top *= 4
                    face_right *= 4
                    face_bottom *= 4
                    face_left *= 4

                    # Check if the center of the face is inside the person's box
                    face_center_x = (face_left + face_right) // 2
                    face_center_y = (face_top + face_bottom) // 2

                    if (
                        person_box_left <= face_center_x <= person_box_right
                        and person_box_top <= face_center_y <= person_box_bottom
                    ):
                        # 5. Use robust face matching for the associated face
                        if self.known_face_encodings:
                            face_distances = face_recognition.face_distance(self.known_face_encodings, face_encoding)
                            best_match_index = np.argmin(face_distances)

                            if face_distances[best_match_index] < self.face_tolerance:
                                name = self.known_face_names[best_match_index]
                                is_known = True

                        # Once a face is matched to this person, stop checking other faces
                        break

                # 6. Update counter and draw labels
                if not is_known:
                    unknown_person_count += 1
                    color = (0, 0, 255)  # Red for Unknown
                    # label = f"Unknown ({conf:.2f})"
                    label = f"Unknown"
                else:
                    color = (0, 255, 0)  # Green for Known
                    label = f"{name}"
                    # label = f"{name} ({conf:.2f})"

                # annotator.box_label(box, label, color=color)

                # annotator.visioneye(box, self.vision_point)
                # build base label from the existing adjust_box_label()
                base_label = self.adjust_box_label(int(cls), float(conf) if conf is not None else 0.0, t_id)

                # custom label for 'person' class (COCO id 0). Use CFG override if provided.
                if int(cls) == 0:
                    prefix = str(self.CFG.get("person_label_prefix", label))
                    custom_label = f"{prefix}:"
                    # if base_label exists, concat both for full display
                    final_label = f"{custom_label} {base_label}" if base_label else custom_label
                else:
                    final_label = base_label

                # draw final label and vision eye mapping
                annotator.box_label(box, label=final_label, color=colors(int(t_id), True))
            else:
                # For non-person classes, use default labeling
                annotator.box_label(box, label=self.adjust_box_label(cls, conf, t_id), color=colors(int(t_id), True))

            annotator.visioneye(box, self.vision_point)

        # 7. Trigger alarm based on the COUNT of unknown people and the 'records' threshold
        if unknown_person_count >= self.records:
            # Schedule play_sound asynchronously (uses asyncio.to_thread when an event loop is running,
            # otherwise falls back to a daemon thread). This avoids blocking the main detection loop.
            try:
                loop = asyncio.get_running_loop()
                loop.create_task(asyncio.to_thread(self.play_sound))
            except RuntimeError:
                # No running asyncio loop (common in regular scripts), use a background thread
                threading.Thread(target=self.play_sound, daemon=True).start()
            except Exception as e:
                LOGGER.exception("Failed to schedule play_sound asynchronously: %s", e)
                # As a last resort, call synchronously (play_sound is idempotent)
                self.play_sound()
        else:
            self.reset_sound()

        plot_im = annotator.result()
        self.display_output(plot_im)

        # Display track count on the frame
        total_tracks = len(getattr(self, "track_ids", []))
        cv2.putText(plot_im, f"Tracks: {total_tracks}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        return SolutionResults(plot_im=plot_im, total_tracks=len(self.track_ids))


# cap = cv2.VideoCapture("../media_files/ruhama.mp4")
# cap = cv2.VideoCapture("../media_files/istockphoto-2002563994-640_adpp_is.mp4")
cap = cv2.VideoCapture("../media_files/WIN_20260227_22_00_29_Pro.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("visioneye_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize vision eye object
# visioneye = solutions.VisionEye(
#     show=True,  # display the output
#     model="yolo12x.pt",  # use any model that Ultralytics supports, e.g., YOLOv10
#     # classes=[0, 2],  # generate visioneye view for specific classes
#     vision_point=(50, 50),  # the point where VisionEye will view objects and draw tracks
# )
AiSecurityGuard = AiSecurityGuard(
    show=True,  # display the output
    model="yolo26m-pose.pt",  # use any model that Ultralytics supports, e.g., YOLOv10
    # classes=[0, 2],  # generate visioneye view for specific classes
    # vision_point=(50, 50),  # the point where VisionEye will view objects and draw tracks
    # conf=0.3,
    # iou=0.5,
    # verbose=True,
    # model="yolo26m-pose.pt",  # use any model that Ultralytics supports, e.g., YOLOv10
    classes=[0, 2],  # generate visioneye view for specific classes
    vision_point=(w // 2 - 250, h - 10),  # the point where VisionEye will view objects and draw tracks
    conf=0.2,
    iou=0.7,
    # persist=True,  # persist tracks even when objects are not detected in the current frame
    tracker="bytetrack.yaml",
    known_face_encodings=known_face_encodings,
    known_face_names=known_face_names,
    records=1,
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = AiSecurityGuard(im0)

    print(results)  # access the output

    video_writer.write(results.plot_im)  # write the video file

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
import cv2

from ultralytics import solutions

import cv2
# from numpy import source

# from ultralytics.utils.plotting import Annotator

import os

# import cv2
import numpy as np
import face_recognition
import pygame

# from ultralytics import solutions
# from ultralytics import YOLO
# from ultralytics.solutions.config import SolutionConfig
from ultralytics.utils import LOGGER

from ultralytics.solutions.solutions import SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors

# ========== 🔊 SOUND SETUP ==========
pygame.mixer.init()
ALARM_FILE = "../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"
if os.path.exists(ALARM_FILE):
    pygame.mixer.music.load(ALARM_FILE)
else:
    print(f"[WARNING] Alarm file '{ALARM_FILE}' not found.")


# ========== 🧠 KNOWN FACE ENCODING LOADER ==========
KNOWN_FACE_DIR = "../family_members/"
known_face_encodings, known_face_names = [], []

if os.path.exists(KNOWN_FACE_DIR):
    for name in os.listdir(KNOWN_FACE_DIR):
        person_dir = os.path.join(KNOWN_FACE_DIR, name)
        if not os.path.isdir(person_dir):
            continue
        for filename in os.listdir(person_dir):
            path = os.path.join(person_dir, filename)
            try:
                img = face_recognition.load_image_file(path)
                enc = face_recognition.face_encodings(img)
                if enc:
                    known_face_encodings.append(enc[0])
                    known_face_names.append(name)
                    print(f"[INFO] Loaded face for {name} from {filename}")
            except Exception as e:
                print(f"[ERROR] Failed loading {path}: {e}")
else:
    print("[WARNING] No known_faces directory found.")


class AiSecurityGuard(solutions.VisionEye):
    def __init__(self, *args, known_face_encodings=None, known_face_names=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.known_face_encodings = known_face_encodings or []
        self.known_face_names = known_face_names or []
        self.sound_played = False
        # Best practice: Set face recognition tolerance during initialization
        self.face_tolerance = 0.55
        self.vision_point = self.CFG["vision_point"]
        self.records = self.CFG.get("records", 1)
        # self.show = self.CFG.get("show", True)

    def play_sound(self):
        """Plays the alarm sound if it's not already playing."""
        if not self.sound_played:
            if pygame.mixer.get_init() and not pygame.mixer.music.get_busy():
                pygame.mixer.music.play()
                self.sound_played = True
                LOGGER.info("🚨 Alarm Triggered: Unknown person count reached threshold.")

    def reset_sound(self):
        """Stops the alarm sound and resets the state."""
        if self.sound_played:
            if pygame.mixer.get_init():
                pygame.mixer.music.stop()
            self.sound_played = False
            LOGGER.info("🟢 Alarm Reset: Area clear.")

    def __call__(self, im0):
        """
        Processes a single frame for person detection and face recognition.
        This implementation follows best practices for accuracy and performance.
        """
        # 1. Get person detections from the base class
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, self.line_width)

        unknown_person_count = 0

        # 2. Optimize by finding all faces in the frame at once (on a smaller version)
        # This is much faster than processing crops for each person.
        h, w, _ = im0.shape
        small_frame = cv2.resize(im0, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        # 3. Iterate through detected PERSONS from YOLO
        for cls, t_id, box, conf in zip(self.clss, self.track_ids, self.boxes, self.confs):
            if int(cls) == 0:  # Skip if not a person
                name = "Unknown"
                is_known = False

                # 4. Associate faces with person boxes
                # Check if any detected face is inside this person's bounding box
                person_box_left, person_box_top, person_box_right, person_box_bottom = map(int, box)

                for (face_top, face_right, face_bottom, face_left), face_encoding in zip(
                    face_locations, face_encodings
                ):
                    # Scale face locations back to original image size
                    face_top *= 4
                    face_right *= 4
                    face_bottom *= 4
                    face_left *= 4

                    # Check if the center of the face is inside the person's box
                    face_center_x = (face_left + face_right) // 2
                    face_center_y = (face_top + face_bottom) // 2

                    if (
                        person_box_left <= face_center_x <= person_box_right
                        and person_box_top <= face_center_y <= person_box_bottom
                    ):
                        # 5. Use robust face matching for the associated face
                        if self.known_face_encodings:
                            face_distances = face_recognition.face_distance(self.known_face_encodings, face_encoding)
                            best_match_index = np.argmin(face_distances)

                            if face_distances[best_match_index] < self.face_tolerance:
                                name = self.known_face_names[best_match_index]
                                is_known = True

                        # Once a face is matched to this person, stop checking other faces
                        break

                # 6. Update counter and draw labels
                if not is_known:
                    unknown_person_count += 1
                    color = (0, 0, 255)  # Red for Unknown
                    # label = f"Unknown ({conf:.2f})"
                    label = "Unknown"
                else:
                    color = (0, 255, 0)  # Green for Known
                    label = f"{name}"
                    # label = f"{name} ({conf:.2f})"

                # annotator.box_label(box, label, color=color)

                # annotator.visioneye(box, self.vision_point)
                # build base label from the existing adjust_box_label()
                base_label = self.adjust_box_label(int(cls), float(conf) if conf is not None else 0.0, t_id)

                # custom label for 'person' class (COCO id 0). Use CFG override if provided.
                if int(cls) == 0:
                    prefix = str(self.CFG.get("person_label_prefix", label))
                    custom_label = f"{prefix}:"
                    # if base_label exists, concat both for full display
                    final_label = f"{custom_label} {base_label}" if base_label else custom_label
                else:
                    final_label = base_label

                # draw final label and vision eye mapping
                annotator.box_label(box, label=final_label, color=color)
            else:
                # For non-person classes, use default labeling
                annotator.box_label(box, label=self.adjust_box_label(cls, conf, t_id), color=colors(int(t_id), True))

            annotator.visioneye(box, self.vision_point)

        # 7. Trigger alarm based on the COUNT of unknown people and the 'records' threshold
        if unknown_person_count >= self.records:
            self.play_sound()
        else:
            self.reset_sound()

        plot_im = annotator.result()
        self.display_output(plot_im)

        # Display track count on the frame
        total_tracks = len(getattr(self, "track_ids", []))
        cv2.putText(plot_im, f"Tracks: {total_tracks}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        return SolutionResults(plot_im=plot_im, total_tracks=len(self.track_ids))


# cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
# cap = cv2.VideoCapture("../media_files/WIN_20251103_14_11_20_Pro.mp4")
cap = cv2.VideoCapture("../media_files/WIN_20260227_22_00_29_Pro.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("visioneye_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize vision eye object
# visioneye = solutions.VisionEye(
#     show=True,  # display the output
#     model="yolo12x.pt",  # use any model that Ultralytics supports, e.g., YOLOv10
#     # classes=[0, 2],  # generate visioneye view for specific classes
#     vision_point=(50, 50),  # the point where VisionEye will view objects and draw tracks
# )
ai_security_guard_instance = AiSecurityGuard(
    show=True,  # display the output
    model="yolo26m-pose.pt",  # use any model that Ultralytics supports, e.g., YOLOv10
    classes=[0, 2],  # generate visioneye view for specific classes
    # vision_point=(50, 50),  # the point where VisionEye will view objects and draw tracks
    conf=0.2,
    iou=0.8,
    # verbose=True,
    # persist=True,
    tracker="bytetrack.yaml",
    known_face_encodings=known_face_encodings,
    known_face_names=known_face_names,
    records=1,
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = ai_security_guard_instance(im0)

    print(results)  # access the output

    video_writer.write(results.plot_im)  # write the video file

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
# Run a workout example
!yolo solutions workout show=True

# Pass a source video
!yolo solutions workout source="./solution_ci_pose_demo.mp4"

# Use keypoints for pushups
!yolo solutions workout kpts="[6, 8, 10]"


In [ ]:
import os
import cv2
import numpy as np
import face_recognition
from ultralytics import solutions
from ultralytics.utils import LOGGER
import pygame
from pathlib import Path  # Fix: Import the Path object
from ultralytics.solutions.solutions import BaseSolution, SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors
# --- 1. Pygame and Known Faces Setup (Global) ---

# Initialize alarm sound
pygame.mixer.init()
alarm_file = "../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"
if os.path.exists(alarm_file):
    pygame.mixer.music.load(alarm_file)
else:
    print(f"⚠️ Warning: Alarm file '{alarm_file}' not found — please check the path.")

# Define the directory for known faces
KNOWN_FACES_DIR = "../family_members/"

# --- 2. Extended Class Definitions ---


# --- FaceRecognitionAlarm Class (Consolidated logic) ---
class FaceRecognitionAlarm(
    solutions.SecurityAlarm
):  # Inherit from SecurityAlarm, not SoundAlarm now for cleaner override
    """
    A security alarm that uses face recognition to trigger alerts only for unknown persons.
    Optimized to run face recognition intermittently for better performance.
    """

    def __init__(self, face_data_path, **kwargs):
        super().__init__(**kwargs)
        self.known_face_encodings = []
        self.known_face_names = []
        self.face_data_path = face_data_path
        self.sound_played = False  # Add sound state here for alarm control
        self.vision_point = self.CFG["vision_point"]

        # Optimization attributes
        self.frame_count = 0
        self.recognition_interval = 5  # Process face recognition every 5 frames
        self.tracked_faces = {}  # Stores recognition state for each track_id: {'name': str, 'cooldown': int}
        self.recognition_cooldown = 15  # Frames to wait before re-checking a face

        self._load_known_faces()

    def _load_known_faces(self):
        """Loads face encodings from a directory with a 'person_name/image.jpg' structure."""
        LOGGER.info(f"Loading known faces from '{self.face_data_path}'...")
        if not os.path.exists(KNOWN_FACES_DIR):
            LOGGER.warning(f"Known faces directory '{KNOWN_FACES_DIR}' not found.")
            return

        for person_name in os.listdir(KNOWN_FACES_DIR):
            person_path = Path(KNOWN_FACES_DIR) / person_name
            if person_path.is_dir():
                for image_file in person_path.glob("*[.jpg,.jpeg,.png]"):
                    try:
                        image = face_recognition.load_image_file(str(image_file))
                        encodings = face_recognition.face_encodings(image)
                        if encodings:
                            self.known_face_encodings.append(encodings[0])
                            self.known_face_names.append(person_name)
                            LOGGER.info(f"  - Loaded face for '{person_name}' from {image_file.name}")
                        else:
                            LOGGER.warning(f"No face found in {image_file}")
                    except Exception as e:
                        LOGGER.error(f"Error loading {image_file}: {e}")

        if not self.known_face_encodings:
            LOGGER.warning("No known faces loaded. All detected persons will be 'Unknown'.")
        else:
            LOGGER.info(
                f"Successfully loaded {len(self.known_face_encodings)} faces for {len(set(self.known_face_names))} people."
            )

    def _get_face_encodings(self, im0):
        """Detects faces and computes encodings, but only on interval frames."""
        if self.frame_count % self.recognition_interval == 0:
            small_frame = cv2.resize(im0, (0, 0), fx=0.25, fy=0.25)
            rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
            face_locations = face_recognition.face_locations(rgb_small_frame)
            face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
            return face_locations, face_encodings
        return [], []

    def _handle_alarm_logic(self, unknown_person_count):
        """Manages the triggering and resetting of the sound and email alarm."""
        if unknown_person_count >= self.records:
            if not self.email_sent:
                # self.send_email(im0, unknown_person_count) # Uncomment if email is setup
                self.email_sent = True
                LOGGER.info(f"📧 Email alert condition met for {unknown_person_count} unknown person(s).")
            if not self.sound_played:
                if pygame.mixer.get_init() and not pygame.mixer.music.get_busy():
                    LOGGER.info("🚨 Playing security alarm!")
                    pygame.mixer.music.play()
                    self.sound_played = True
        elif self.email_sent or self.sound_played:
            self.email_sent = False
            self.sound_played = False
            LOGGER.info("🟢 Alarm system reset.")
            if pygame.mixer.get_init():
                pygame.mixer.music.stop()

    def process(self, im0):
        """Overrides process to add optimized face recognition and alarm logic."""
        self.frame_count += 1
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, line_width=self.line_width)

        unknown_person_count = 0
        person_cls_id = 0  # COCO class ID for 'person'

        face_locations, face_encodings = self._get_face_encodings(im0)

        # Process each tracked object
        for i, (box, conf, cls, track_id) in enumerate(zip(self.boxes, self.confs, self.clss, self.track_ids)):
            cls = self.clss[i]
            label = self.names[cls]
            color = colors(cls, True)

            if cls == person_cls_id:
                if track_id not in self.tracked_faces or self.tracked_faces[track_id]["cooldown"] == 0:
                    if face_encodings:
                        x1, y1, x2, y2 = map(int, box)
                        found_match = False
                        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
                            top, right, bottom, left = top * 4, right * 4, bottom * 4, left * 4
                            face_center_x, face_center_y = (left + right) // 2, (top + bottom) // 2

                            if x1 <= face_center_x <= x2 and y1 <= face_center_y <= y2:
                                matches = face_recognition.compare_faces(
                                    self.known_face_encodings, face_encoding, tolerance=0.55
                                )
                                name = "Unknown"
                                if True in matches:
                                    best_match_index = np.argmin(
                                        face_recognition.face_distance(self.known_face_encodings, face_encoding)
                                    )
                                    if matches[best_match_index]:
                                        name = self.known_face_names[best_match_index]

                                self.tracked_faces[track_id] = {"name": name, "cooldown": self.recognition_cooldown}
                                found_match = True
                                break

                        if not found_match:
                            self.tracked_faces[track_id] = {"name": "No Face", "cooldown": self.recognition_cooldown}

                if track_id in self.tracked_faces:
                    face_info = self.tracked_faces[track_id]
                    name = face_info["name"]
                    if name == "Unknown":
                        label, color = "Unknown (ALARM!)", (0, 0, 255)
                        unknown_person_count += 1
                    elif name == "No Face":
                        label, color = "Person (No Face)", (255, 192, 203)  # Pink
                    else:
                        label, color = f"{name} (Known)", (0, 255, 0)

                    if face_info["cooldown"] > 0:
                        self.tracked_faces[track_id]["cooldown"] -= 1

            annotator.box_label(box, label=label, color=color)

            # build base label from the existing adjust_box_label()
            base_label = self.adjust_box_label(int(cls), float(conf) if conf is not None else 0.0, track_id)

            # custom label for 'person' class (COCO id 0). Use CFG override if provided.
            if int(cls) == 0:
                prefix = str(self.CFG.get("person_label_prefix", label))
                custom_label = f"{prefix}:"
                # if base_label exists, concat both for full display
                final_label = f"{custom_label} {base_label}" if base_label else custom_label
            else:
                final_label = base_label

            # draw final label and vision eye mapping
            annotator.box_label(box, label=final_label, color=color)
        else:
            # For non-person classes, use default labeling
            annotator.box_label(
                box, label=self.adjust_box_label(cls, conf, track_id), color=colors(int(track_id), True)
            )

        annotator.visioneye(box, self.vision_point)

        self._handle_alarm_logic(unknown_person_count)

        plot_im = annotator.result()
        self.display_output(plot_im)

        # Return the SolutionResults with the correct flags and counts
        return SolutionResults(
            plot_im=plot_im,
            # im0=im0,
            total_tracks=len(getattr(self, "track_ids", [])),
            email_sent=self.email_sent,
            sound_played=self.sound_played,
        )


# --- 3. Main Execution Block ---

# Open video
cap = cv2.VideoCapture("../media_files/WIN_20260227_22_00_29_Pro.mp4")
# cap = cv2.VideoCapture("media_files/theaf_surveillance/1093628701-preview.mp4")
assert cap.isOpened(), "❌ Error: Cannot read video file."

# Video writer setup
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("security_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Security Alarm setup: USE THE EXTENDED CLASS
# records=3 means alarm will trigger if 3 or more UNKNOWN persons are detected.
securityalarm = FaceRecognitionAlarm(
    show=True,
    model="yolo26m-pose.pt",  # Use a valid and fast model
    records=1,
    # classes=[0, 2],        # Only detect 'person' (ID 0) and 'bicycle' (ID 2) for face recognition
    face_data_path=KNOWN_FACES_DIR,
    conf=0.1,
    iou=0.9,
    tracker="bytetrack.yaml",
)

# Optional: Email setup
from_email = "deveansari@gmail.com"
password = "ddgl yjef dlaw tuzg"
to_email = "rahatansari.tpu@gmail.com"

# securityalarm.authenticate(from_email, password, to_email) # Uncomment and fix password/email

# --- PROCESS VIDEO ---
print("\n--- Starting Video Processing. Press 'q' to terminate. ---")
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("✅ Video processing completed.")
        break

    # Run Detection and Alarm Logic
    results = securityalarm(frame)
    video_writer.write(results.plot_im)  # write the processed frame.
    cv2.imshow("Face Recognition Security Alarm", results.plot_im)

    # Check for 'q' key press to terminate
    if cv2.waitKey(1) & 0xFF == ord("q"):
        print("🛑 Termination key 'q' pressed. Stopping...")
        break

    # Allow pygame to process events
    # pygame.event.pump()

# Cleanup
cap.release()
video_writer.release()
cv2.destroyAllWindows()
pygame.mixer.quit()


In [ ]:
# Persist=True maintains the ID of each animal
results = model.track(source="pasture_video.mp4", persist=True, tracker="bytetrack.yaml")


In [ ]:
from ultralytics import YOLO

# Load the YOLO26 model
model = YOLO("yolo26n.pt")

# Run inference on an image
results = model("https://ultralytics.com/images/bus.jpg")

# Access bounding box coordinates (xyxy format) for the first detected object
boxes = results[0].boxes
print(boxes.xyxy[0])  # Output: tensor([x1, y1, x2, y2, ...])
